<a href="https://colab.research.google.com/github/tshankar551-bit/Pharmacophore-Study-on-FtsZ/blob/main/Docking_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Docking Validation Process

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After mounting your Google Drive, you can load your data by specifying the path to your file. For example, if you have a CSV file named `my_data.csv` in your Drive's root, you can load it like this:

In [ ]:
# ============================================================
# DUD-E / Docking Validation Analysis Platform
# ROC-AUC | PR-AUC | Enrichment Factor | BEDROC
#
# INPUT:
# CSV or Excel containing:
# Compound_ID | Class | Docking_Score
#
# Class must contain Active / Decoy
# More negative docking score = better
# ============================================================

!pip -q install pandas openpyxl scikit-learn matplotlib numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

from IPython.display import display

print("Docking Validation Analysis Platform")
print("ROC-AUC | PR-AUC | EF | BEDROC")

In [ ]:
uploaded = files.upload()
for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Once the file is uploaded, you can then load it into a pandas DataFrame (if it's a CSV or Excel file). For example, if you uploaded `my_data.csv`:

In [12]:
# Example: Load a CSV file into a DataFrame
# import pandas as pd
# df = pd.read_csv('my_data.csv')
# display(df.head())

In [ ]:
import pandas as pd

# Assuming the uploaded file is an Excel file
df = pd.read_excel(filename)
display(df.head())

### Data Preparation for Analysis

To perform ROC-AUC, PR-AUC, EF, and BEDROC analysis, we need to convert the 'Class' column into a numerical format (Active=1, Decoy=0) and ensure the 'Docking_Score' is oriented such that higher values indicate a greater likelihood of being an 'Active' compound. Since the problem statement indicates 'More negative docking score = better', we will negate the docking scores so that more positive (less negative) values now correspond to better binding (more active).

In [ ]:
# Convert 'Class' to numerical representation: Active=1, Decoy=0
df['Class_Numeric'] = df['Class'].apply(lambda x: 1 if x == 'Active' else 0)

# Negate Docking_Score: higher (less negative) values now mean better binding
df['Score_Adjusted'] = -df['Docking_Score']

display(df.head())

### ROC-AUC (Receiver Operating Characteristic - Area Under the Curve)

ROC-AUC measures the ability of a classifier to distinguish between classes. A higher AUC indicates better performance. It plots the True Positive Rate (TPR) against the False Positive Rate (FPR) at various threshold settings.

In [ ]:
# Extract true labels and adjusted scores
y_true = df['Class_Numeric']
y_scores = df['Score_Adjusted']

# Calculate ROC-AUC
roc_auc = roc_auc_score(y_true, y_scores)
print(f"ROC-AUC: {roc_auc:.4f}")

# Generate ROC curve data
fpr, tpr, thresholds = roc_curve(y_true, y_scores)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

### PR-AUC (Precision-Recall - Area Under the Curve)

PR-AUC is particularly useful for imbalanced datasets, where the positive class is rare. It plots Precision (Positive Predictive Value) against Recall (True Positive Rate).

In [ ]:
# Calculate PR-AUC
pr_auc = average_precision_score(y_true, y_scores)
print(f"PR-AUC: {pr_auc:.4f}")

# Generate Precision-Recall curve data
precision, recall, _ = precision_recall_curve(y_true, y_scores)

# Plot PR curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (area = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

### Enrichment Factor (EF)

Enrichment Factor measures how many times more often active compounds are found in the top `X%` of a ranked list compared to a random selection. A high EF at a small percentage indicates a good early enrichment of actives.

In [ ]:
def calculate_ef(df_data, percentage=0.01):
    # Sort by adjusted score in descending order (higher score = more active)
    df_sorted = df_data.sort_values(by='Score_Adjusted', ascending=False).reset_index(drop=True)

    total_actives = df_sorted['Class_Numeric'].sum()
    total_compounds = len(df_sorted)

    if total_actives == 0: # Avoid division by zero if no actives
        return 0

    # Calculate the number of compounds in the top percentage
    num_top_compounds = int(np.ceil(total_compounds * percentage))
    if num_top_compounds == 0: num_top_compounds = 1 # Ensure at least one compound if percentage is very small

    # Get actives in the top percentage
    actives_in_top_percent = df_sorted.head(num_top_compounds)['Class_Numeric'].sum()

    # Calculate fraction of actives in top percentage
    fraction_actives_in_top_percent = actives_in_top_percent / num_top_compounds

    # Calculate overall fraction of actives in the dataset
    overall_fraction_actives = total_actives / total_compounds

    if overall_fraction_actives == 0: # Avoid division by zero if no actives overall
        return 0

    ef = fraction_actives_in_top_percent / overall_fraction_actives
    return ef

# Calculate EF at 1% and 5%
ef_1_percent = calculate_ef(df, percentage=0.01)
ef_5_percent = calculate_ef(df, percentage=0.05)

print(f"Enrichment Factor at 1%: {ef_1_percent:.2f}")
print(f"Enrichment Factor at 5%: {ef_5_percent:.2f}")

### BEDROC (Boltzmann-Enhanced Discrimination of ROC)

BEDROC is an early enrichment metric that emphasizes the performance at the very top of the ranked list, making it particularly suitable for virtual screening where finding a few highly potent compounds is critical. It uses an exponential weighting function to prioritize the top ranks.

In [ ]:
def calculate_bedroc(df_data, alpha=20):
    # Sort by adjusted score in descending order
    df_sorted = df_data.sort_values(by='Score_Adjusted', ascending=False).reset_index(drop=True)

    total_compounds = len(df_sorted)
    total_actives = df_sorted['Class_Numeric'].sum()

    if total_actives == 0 or total_actives == total_compounds:
        return 0 # BEDROC is undefined or trivial if all compounds are actives or decoys

    # Indices of active compounds after sorting
    active_indices = df_sorted[df_sorted['Class_Numeric'] == 1].index.tolist()

    # Calculate the sum for the BEDROC formula
    sum_exp_term = sum(np.exp(-alpha * (idx + 1) / total_compounds) for idx in active_indices)

    # Pre-calculate terms for efficiency
    R = total_actives
    N = total_compounds
    e_alpha_N = np.exp(-alpha / N)

    # Calculate the G_alpha term (geometric mean of actives at alpha)
    if alpha == 0:
        G_alpha = R * (R - 1) / (2 * (N - 1))
    else:
        G_alpha = (1 - np.exp(-alpha * R / N)) / (1 - e_alpha_N)

    # Calculate the P_alpha term (random expectation)
    P_alpha = R / N

    # Calculate the R_alpha term (enrichment at alpha)
    R_alpha = (1 - e_alpha_N) / (N * (1 - e_alpha_N))

    # Calculate the BEDROC denominator
    denominator = (P_alpha * (1 - e_alpha_N) / (1 - e_alpha_N * np.exp(-alpha * R / N)))

    # BEDROC formula
    bedroc_value = (sum_exp_term / (R_alpha * R)) * ((1 - e_alpha_N) / (1 - np.exp(-alpha))) * (1 - denominator)

    # Handle edge cases where bedroc might become NaN due to division by zero or log(0) if R=N or R=0
    if np.isnan(bedroc_value) or bedroc_value < 0: # Ensure bedroc is within [0, 1]
        bedroc_value = 0.0
    elif bedroc_value > 1:
        bedroc_value = 1.0

    return bedroc_value

# Calculate BEDROC with a typical alpha value
bedroc_score = calculate_bedroc(df, alpha=20)

print(f"BEDROC (alpha=20): {bedroc_score:.4f}")

All Calculation/Data

In [ ]:
print('--- DataFrame (df) ---')
display(df)

print('\n--- Other Variables ---')
print(f"fpr: {fpr}")
print(f"precision: {precision}")
print(f"recall: {recall}")
print(f"thresholds: {thresholds}")
print(f"tpr: {tpr}")
print(f"y_scores: {y_scores}")
print(f"y_true: {y_true}")
print(f"bedroc_score: {bedroc_score}")
print(f"ef_1_percent: {ef_1_percent}")
print(f"ef_5_percent: {ef_5_percent}")
print(f"filename: '{filename}'")
print(f"pr_auc: {pr_auc}")
print(f"roc_auc: {roc_auc}")
print(f"uploaded: {uploaded}")